In [1]:
%load_ext autoreload
%autoreload 2

# Import

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Union
from glob import glob

import ipywidgets as widgets
from IPython.display import display

import sys
PHASE1_SRC = Path("../Phase1/src/")
if str(PHASE1_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE1_SRC))

from space import Space, PLASMA_ENV
from reward import reward_peak, reward_latent, reward_peak_new
from machinelearning import train_single_model, predict_with_model, preprocess, build_models
from VEM import oracle_fn, TARGET_NAMES
from plotting import *

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from umap import UMAP
from sklearn.decomposition import PCA

# Loading

In [3]:
run = "al_results_cplasma"  # switchable to "grid_results_mix_and_match"

In [4]:
grid = pd.read_csv(f"{run}/grid_summary.csv")
grid = grid.rename(columns={"sampling_strategy": "schedule"})
grid["schedule"] = grid["schedule"].apply(eval)

REWARD_FILTER = "reward_peak_new"  # "reward_latent" | "reward_peak" | "reward_peak_new"
grid = grid[grid["reward_fn_name"] == REWARD_FILTER]
grid[["run", "schedule", "n_init", "n_candidates_per_iter", "n_iterations", "best_reward", "mean_reward"]]

,run,schedule,n_init,n_candidates_per_iter,n_iterations,best_reward,mean_reward
0,run_0,{'gflownet': 6},60,10,6,0.873705,0.425723
1,run_1,{'random': 6},60,10,6,0.981646,0.462222
2,run_2,{'lhs': 6},60,10,6,0.991242,0.469917
3,run_3,{'grid': 6},60,10,6,0.963652,0.464983
4,run_4,{'gp': 6},60,10,6,0.990809,0.502933
5,run_5,{'genetic': 6},60,10,6,0.997777,0.565717
6,run_6,"{'lhs': 3, 'gflownet': 3}",60,10,6,0.889611,0.449313
7,run_7,"{'lhs': 3, 'genetic': 3}",60,10,6,0.980388,0.539270


In [5]:
paths = sorted(glob(f"{run}/*/al_metrics.csv"))
paths = [p for p in paths if Path(p).parent.name in grid["run"].values]
if not paths:
    raise ValueError(f"No files found at {run}/*/al_metrics.csv")
df = pd.concat([load(p).assign(run=Path(p).parent.name) for p in paths], ignore_index=True)

In [6]:
df.head()

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,best_reward_this_iter,mean_reward_this_iter,best_predicted_reward,gfn_logZ,elapsed_sec,run
0,1,17:07:15,70,70,10,0.873705,0.414741,0.269312,0.078009,gflownet,0.735558,0.454655,0.495319,35.806004,53.6,run_0
1,2,17:08:01,80,80,10,0.873705,0.420547,0.438054,0.076250,gflownet,0.727645,0.461184,0.581430,35.582916,45.9,run_0
2,3,17:08:27,90,90,10,0.873705,0.419806,0.332147,0.076901,gflownet,0.609534,0.413884,0.618660,35.353794,25.7,run_0
3,4,17:08:52,100,100,10,0.873705,0.423423,0.456359,0.078325,gflownet,0.685858,0.455976,0.608395,35.116459,25.4,run_0
4,5,17:09:18,110,110,10,0.873705,0.424266,0.322850,0.067497,gflownet,0.539519,0.432693,0.658848,34.874935,26.4,run_0


In [7]:
print("GFlowNet iterations recorded:", int(df[df["sampling_strategy"] == "gflownet"].shape[0]))

GFlowNet iterations recorded: 9


In [8]:
best_runs(df)

,run,final_mean_reward,final_best_reward,final_proxy_r2,final_proxy_mae,total_oracle_evals,sampling_strategy
0,run_5,0.565717,0.997777,0.791418,0.046782,120,genetic
1,run_6,0.449313,0.889611,0.523298,0.066590,120,gflownet
2,run_4,0.502933,0.990809,0.761266,0.063894,120,gp
3,run_3,0.464983,0.963652,0.638315,0.072647,120,grid
4,run_2,0.469917,0.991242,0.555266,0.069678,120,lhs
5,run_1,0.462222,0.981646,0.572198,0.070931,120,random


## Visualization

In [9]:
metrics = grid.merge(
    df[[c for c in df.columns if c != "elapsed_sec"]],
    on="run",
    how="left",
)

In [10]:
def _schedule_str(sched: dict) -> str:
    if isinstance(sched, str):
        sched = eval(sched)
    return "→".join(f"{k}:{v}" for k, v in sched.items())

def _n_phases(sched: dict) -> int:
    if isinstance(sched, str):
        sched = eval(sched)
    return len(sched)

metrics["schedule_str"] = metrics["schedule"].map(_schedule_str)
metrics["n_phases"] = metrics["schedule"].map(_n_phases)
metrics["is_pure"] = metrics["n_phases"] == 1

In [11]:
pure = metrics[metrics["is_pure"]]
mixed = metrics[~metrics["is_pure"]]

In [12]:
select_best_runs(mixed, metric="best_reward", groupby="schedule_str")[
    ["run", "schedule_str", "iteration", "best_reward", "mean_reward"]
].drop_duplicates(subset=["run"], keep="last")

,run,schedule_str,iteration,best_reward,mean_reward
47,run_7,lhs:3→genetic:3,6,0.980388,0.539270
41,run_6,lhs:3→gflownet:3,6,0.889611,0.449313


In [13]:
print("Runs per schedule:")
display(metrics.groupby("schedule_str")["run"].nunique().to_frame("n_runs"))

Runs per schedule:


,n_runs
schedule_str,
genetic:6,1
gflownet:6,1
gp:6,1
grid:6,1
lhs:3→genetic:3,1
lhs:3→gflownet:3,1
lhs:6,1
random:6,1


Summary of pure and mixed schedules (reward curves, scatter tradeoffs, variance, efficiency, surrogate accuracy):

In [14]:
select_best_runs(pure, metric="best_reward")[["run", "sampling_strategy", "best_reward", "mean_reward"]].drop_duplicates(subset=["run"], keep="last")

,run,sampling_strategy,best_reward,mean_reward
35,run_5,genetic,0.997777,0.565717
5,run_0,gflownet,0.873705,0.425723
29,run_4,gp,0.990809,0.502933
23,run_3,grid,0.963652,0.464983
17,run_2,lhs,0.991242,0.469917
11,run_1,random,0.981646,0.462222


In [15]:
select_best_runs(mixed, metric="best_reward", groupby="schedule_str")[["run", "schedule_str", "best_reward", "mean_reward"]].drop_duplicates(subset=["run"], keep="last")

,run,schedule_str,best_reward,mean_reward
47,run_7,lhs:3→genetic:3,0.980388,0.539270
41,run_6,lhs:3→gflownet:3,0.889611,0.449313


In [16]:
plot_reward_curves(select_best_runs(pure, metric="mean_reward"), metric="mean_reward_this_iter", title="Mean reward per iteration (pure strategies)", ceiling=True)

In [17]:
agg, symbols = aggregate_seeds_for_scatter(pure)
agg.head()

,sampling_strategy,acquisition,n_candidates_per_iter,n_init,n_iterations,init_method,gfn_batch_size,gfn_n_train_steps,schedule_str,best_reward,best_reward_std,mean_reward,mean_reward_std,n,run,marker_symbol
0,genetic,top_k,10,60,6,random,16,150,genetic:6,0.986906,0.011521,0.527786,0.039062,6,60 + 10 * 6 | acquisition: top_k | init: random,0
1,gflownet,top_k,10,60,6,random,16,150,gflownet:6,0.873705,0.000000,0.421418,0.003965,6,60 + 10 * 6 | acquisition: top_k | init: rando...,0
2,gp,top_k,10,60,6,random,16,150,gp:6,0.978521,0.009518,0.473381,0.024810,6,60 + 10 * 6 | acquisition: top_k | init: random,0
3,grid,top_k,10,60,6,random,16,150,grid:6,0.920792,0.035745,0.449463,0.014431,6,60 + 10 * 6 | acquisition: top_k | init: random,0
4,lhs,top_k,10,60,6,random,16,150,lhs:6,0.903899,0.043260,0.451319,0.016483,6,60 + 10 * 6 | acquisition: top_k | init: random,0


In [18]:
plot_scatter(agg, symbols)

In [19]:
agg_m, symbols_m = aggregate_seeds_for_scatter(mixed)
fig2 = plot_scatter(agg_m, symbols_m)
fig2.show()

In [20]:
out1 = widgets.Output(layout={'border': '1px solid black', 'width': '50%'})
out2 = widgets.Output(layout={'border': '1px solid black', 'width': '50%'})

with out1:
    agg_p, symbols_p = aggregate_seeds_for_scatter(pure)
    fig1 = plot_scatter(agg_p, symbols_p)
    fig1.show()

with out2:
    agg_m2, symbols_m2 = aggregate_seeds_for_scatter(mixed)
    fig2b = plot_scatter(agg_m2, symbols_m2)
    fig2b.show()

display(widgets.HBox([out1, out2]))

In [21]:
plot_seed_variance(metrics, metric="mean_reward_this_iter")

In [22]:
plot_oracle_efficiency(metrics, metric="mean_reward_this_iter")

In [23]:
plot_proxy_curves(metrics)

In [24]:
plot_final_boxplot(metrics, metric="best_reward", groupby="schedule_str", color_by="n_phases")

In [25]:
plot_reward_heatmap(metrics, metric="mean_reward", groupby="schedule_str")

In [26]:
plot_budget_tradeoff(metrics, metric="mean_reward", groupby="schedule_str")

In [27]:
embedding, preprocessor, reducer = compute_global_embedding(
    method="umap",
    n_samples=5000,
)
print("Embedded background shape:", embedding.shape)

C:\Users\dimon\Downloads\Project\PhD\.venv\lib\site-packages\umap\umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
C:\Users\dimon\Downloads\Project\PhD\.venv\lib\site-packages\umap\umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedded background shape: (5000, 12)


## Pure strategies only

In [28]:
sampled_dict_pure = load_best_per_schedule(
    pure,
    results_dir=run,
    preprocessor=preprocessor,
    reducer=reducer,
    groupby="sampling_strategy",
)

In [29]:
plot_sampling_grid(sampled_dict_pure, cols=2)

In [30]:
plot_exploration_dashboard(sampled_dict_pure, preprocessor)

In [31]:
plot_peak_coverage(sampled_dict_pure, "inverse", reward="new")

In [32]:
fig, table = plot_peak_coverage_heatmap(sampled_dict_pure, reward="new", metric="fraction")
fig.update_layout(height=480, width=800)
fig.show()
display(table.round(4))

,Peak 1,Peak 2,Peak 3,Peak 4,Peak 5,Weighted
method,,,,,,
genetic,0.9978,0.4637,0.2152,0.0037,0.0227,0.3699
gflownet,0.2166,0.9343,0.9430,0.2315,0.8103,0.5952
gp,0.9908,0.7035,0.9639,0.6850,0.8617,0.8603
grid,0.9637,0.7085,0.9865,0.7811,0.3375,0.7621
lhs,0.9912,0.8510,0.9821,0.9629,0.6078,0.8808
random,0.9816,0.8006,0.9058,0.9229,0.2200,0.7641


## All schedules

In [33]:
top = select_top_runs(metrics, n=41, metric="best_reward", groupby="schedule_str", dedupe=True)

sampled_dict_best = load_best_per_schedule(
    top, results_dir=run, preprocessor=preprocessor, reducer=reducer, groupby="schedule_str"
)
sampled_dict_all = load_pooled_per_schedule(
    top, results_dir=run, preprocessor=preprocessor, reducer=reducer, groupby="schedule_str"
)

In [34]:
schedule_lookup = metrics.drop_duplicates("schedule_str").set_index("schedule_str")["schedule"]
schedules = {name: schedule_lookup[name] for name in sampled_dict_all.keys()}

In [35]:
fig = plot_exploration_dashboard(sampled_dict_all, preprocessor, schedules)
fig.update_layout(width=1000)
fig.show()

In [36]:
t = phase_transition_table(sampled_dict_all, preprocessor, schedules)
display(t.sort_values("reward_mean", ascending=False).round(4))

,schedule,phase,phase_start,phase_end,reward_mean,novelty_mean,batch_diversity_mean,coverage_gap_end,reward_mean_delta,novelty_mean_delta,batch_diversity_mean_delta
5,lhs:3→genetic:3,genetic,4,6,0.7952,1.5859,2.1226,0.6405,0.2496,-0.7473,-1.9885
0,genetic:6,genetic,1,6,0.7233,1.2211,1.5660,0.8502,NaN,NaN,NaN
2,gp:6,gp,1,6,0.5978,2.1689,3.7064,0.6335,NaN,NaN,NaN
4,lhs:3→genetic:3,lhs,1,3,0.5457,2.3333,4.1111,0.6646,NaN,NaN,NaN
6,lhs:3→gflownet:3,lhs,1,3,0.5457,2.3333,4.1111,0.6646,NaN,NaN,NaN
8,lhs:6,lhs,1,6,0.5317,2.3254,4.1387,0.6045,NaN,NaN,NaN
3,grid:6,grid,1,6,0.5219,2.4626,4.4225,0.6377,NaN,NaN,NaN
9,random:6,random,1,6,0.5164,2.2647,3.9638,0.6303,NaN,NaN,NaN
1,gflownet:6,gflownet,1,6,0.4434,1.1450,2.2758,0.6296,NaN,NaN,NaN
7,lhs:3→gflownet:3,gflownet,4,6,0.4354,1.9313,2.2909,0.6146,-0.1103,-0.4020,-1.8202


In [37]:
plot_exploration_heatmap(sampled_dict_all, preprocessor, schedules)

In [38]:
fig, table = plot_peak_coverage_heatmap(sampled_dict_all, reward="new", metric="fraction")
fig.update_layout(height=500, width=850)
fig.show()
display(table.round(4))

,Peak 1,Peak 2,Peak 3,Peak 4,Peak 5,Weighted
method,,,,,,
genetic:6,0.9978,0.4637,0.2152,0.0037,0.0227,0.3699
gflownet:6,0.2166,0.9343,0.9430,0.2315,0.8103,0.5952
gp:6,0.9908,0.7035,0.9639,0.6850,0.8617,0.8603
grid:6,0.9637,0.7085,0.9865,0.7811,0.3375,0.7621
lhs:3→genetic:3,0.9804,0.7049,0.9785,0.8832,0.2592,0.7660
lhs:3→gflownet:3,0.8896,0.9319,0.9785,0.8832,0.9613,0.9278
lhs:6,0.9912,0.8510,0.9821,0.9629,0.6078,0.8808
random:6,0.9816,0.8006,0.9058,0.9229,0.2200,0.7641


In [39]:
df_iter = metrics[metrics["iteration"] > 0]

fig = go.Figure()
for i, val in enumerate(sorted(df_iter["gfn_loss"].dropna().unique())):
    agg_gfn = df_iter[df_iter["gfn_loss"] == val].groupby("iteration")["mean_reward_this_iter"].agg(["mean", "std"]).reset_index()
    c = f"hsl({i * 60}, 70%, 50%)"
    fig.add_trace(go.Scatter(
        x=agg_gfn["iteration"], y=agg_gfn["mean"], name=str(val),
        line=dict(color=c, width=2), mode="lines+markers",
    ))
    if agg_gfn["std"].notna().any():
        std_vals = agg_gfn["std"].fillna(0.0)
        fig.add_trace(go.Scatter(
            x=pd.concat([agg_gfn["iteration"], agg_gfn["iteration"][::-1]]),
            y=pd.concat([agg_gfn["mean"] + std_vals, (agg_gfn["mean"] - std_vals)[::-1]]),
            fill="toself", fillcolor=c, opacity=0.15, line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))
fig.update_layout(
    template="plotly_white",
    title="mean_reward_this_iter by gfn_loss",
    xaxis_title="iteration",
    yaxis_title="mean_reward_this_iter",
    hovermode="x unified",
)
fig.show()

if "gfn_logZ" in df_iter.columns and df_iter["gfn_logZ"].notna().any():
    gfn_only = df_iter[df_iter["gfn_logZ"].notna()]
    fig_logz = px.line(
        gfn_only,
        x="iteration",
        y="gfn_logZ",
        color="schedule_str",
        markers=True,
        title="GFlowNet learned logZ across Active Learning iterations",
        template="plotly_white",
    )
    fig_logz.show()